In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DateType
from pyspark.sql.functions import count, col

In [2]:
problem_name = os.path.abspath(".").split("/")[-1]

In [3]:
spark = SparkSession.builder.appName(problem_name).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 22:05:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("order_id", IntegerType(), True),
    StructField("order_date", DateType(), True)
])

In [5]:
customers = spark.read\
.option("header", "true")\
.format("csv")\
.schema(customers_schema)\
.load("files/customers.txt")

In [6]:
result_df = customers\
.groupBy("customer_id").agg(count("order_id").alias("orders_count"))

In [7]:
result_df.show(truncate=False)

[Stage 0:>                                                          (0 + 1) / 1]

+-----------+------------+
|customer_id|orders_count|
+-----------+------------+
|101        |2           |
|103        |1           |
|102        |2           |
+-----------+------------+



In [8]:
customers.createOrReplaceTempView("customers")

In [9]:
spark.sql(
    """
    select customer_id, count(order_id) as orders_count
    from customers
    group by customer_id
    """
).show(truncate=False)

+-----------+------------+
|customer_id|orders_count|
+-----------+------------+
|101        |2           |
|103        |1           |
|102        |2           |
+-----------+------------+



In [10]:
spark.stop()